# LeetCode #1368: Minimum Cost to Make at Least One Valid Path in a Grid

https://leetcode.com/problems/minimum-cost-to-make-at-least-one-valid-path-in-a-grid/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (Dijkstra with unit costs)** | $O(nm \log(nm))$ | $O(nm)$ |
| **Optimal: 0-1 BFS (Deque) ★** | $O(nm)$ | $O(nm)$ |

---

## Understanding the Methods

### Brute Force (Dijkstra with unit costs)
Treat "follow existing arrow" as cost 0 and "change direction" as cost 1. Run Dijkstra. Correct but carries a $\log$ factor from the priority queue.

### Optimal: 0-1 BFS (Deque) ★
When all edge weights are 0 or 1, a deque replaces the heap: push a neighbour to the **front** if the edge costs 0 (free move), to the **back** if it costs 1 (direction change needed). This achieves $O(nm)$ time — each cell is settled at most once.

**Constraints:**
* $1 \le m, n \le 100$
* Grid values in $\{1, 2, 3, 4\}$ (right, left, down, up)

## Solutions

### C#

In [ ]:
using System.Collections.Generic;

public class Solution {
    public int MinCost(int[][] grid) {
        int rows = grid.Length, cols = grid[0].Length;
        // Arrow directions: 1=right, 2=left, 3=down, 4=up
        int[][] dirs = {{0,1},{0,-1},{1,0},{-1,0}};
        var dist = new int[rows, cols];
        for (int i = 0; i < rows; i++) for (int j = 0; j < cols; j++) dist[i,j] = int.MaxValue;
        dist[0,0] = 0;

        var dq = new LinkedList<(int r, int c)>();
        dq.AddFirst((0, 0));

        while (dq.Count > 0) {
            var (r, c) = dq.First.Value; dq.RemoveFirst();
            for (int d = 0; d < 4; d++) {
                int nr = r + dirs[d][0], nc = c + dirs[d][1];
                if (nr < 0 || nr >= rows || nc < 0 || nc >= cols) continue;
                // Cost 0 if existing arrow already points in this direction, else 1
                int cost = grid[r][c] == d + 1 ? 0 : 1;
                int newDist = dist[r, c] + cost;
                if (newDist < dist[nr, nc]) {
                    dist[nr, nc] = newDist;
                    // Free move → front of deque; direction change → back
                    if (cost == 0) dq.AddFirst((nr, nc));
                    else dq.AddLast((nr, nc));
                }
            }
        }
        return dist[rows - 1, cols - 1];
    }
}

### Python

In [ ]:
from collections import deque

class Solution:
    def minCost(self, grid: list[list[int]]) -> int:
        rows, cols = len(grid), len(grid[0])
        # Directions indexed 1-4: right, left, down, up
        dirs = [(0,1),(0,-1),(1,0),(-1,0)]
        dist = [[float('inf')] * cols for _ in range(rows)]
        dist[0][0] = 0
        dq = deque([(0, 0)])

        while dq:
            r, c = dq.popleft()
            for d, (dr, dc) in enumerate(dirs):
                nr, nc = r + dr, c + dc
                if not (0 <= nr < rows and 0 <= nc < cols):
                    continue
                # Following the existing arrow is free; changing direction costs 1
                cost = 0 if grid[r][c] == d + 1 else 1
                new_dist = dist[r][c] + cost
                if new_dist < dist[nr][nc]:
                    dist[nr][nc] = new_dist
                    if cost == 0:
                        dq.appendleft((nr, nc))  # free move — push to front
                    else:
                        dq.append((nr, nc))       # costs 1 — push to back
        return dist[rows-1][cols-1]

### Go

In [ ]:
func minCost(grid [][]int) int {
	rows, cols := len(grid), len(grid[0])
	dirs := [][2]int{{0,1},{0,-1},{1,0},{-1,0}}
	dist := make([][]int, rows)
	for i := range dist {
		dist[i] = make([]int, cols)
		for j := range dist[i] { dist[i][j] = 1<<31 - 1 }
	}
	dist[0][0] = 0

	// Deque implemented as a slice (front = index 0)
	type pt = [2]int
	dq := []pt{{0,0}}

	for len(dq) > 0 {
		cur := dq[0]; dq = dq[1:]
		r, c := cur[0], cur[1]
		for d, dir := range dirs {
			nr, nc := r+dir[0], c+dir[1]
			if nr < 0 || nr >= rows || nc < 0 || nc >= cols { continue }
			cost := 1
			if grid[r][c] == d+1 { cost = 0 } // arrow already points here
			nd := dist[r][c] + cost
			if nd < dist[nr][nc] {
				dist[nr][nc] = nd
				if cost == 0 {
					dq = append([]pt{{nr,nc}}, dq...) // push front
				} else {
					dq = append(dq, pt{nr,nc}) // push back
				}
			}
		}
	}
	return dist[rows-1][cols-1]
}

### Rust

In [ ]:
use std::collections::VecDeque;

impl Solution {
    pub fn min_cost(grid: Vec<Vec<i32>>) -> i32 {
        let (rows, cols) = (grid.len(), grid[0].len());
        let dirs = [(0i32,1i32),(0,-1),(1,0),(-1,0)];
        let mut dist = vec![vec![i32::MAX; cols]; rows];
        dist[0][0] = 0;
        let mut dq: VecDeque<(usize, usize)> = VecDeque::from([(0, 0)]);

        while let Some((r, c)) = dq.pop_front() {
            for (d, &(dr, dc)) in dirs.iter().enumerate() {
                let nr = r as i32 + dr;
                let nc = c as i32 + dc;
                if nr < 0 || nr >= rows as i32 || nc < 0 || nc >= cols as i32 { continue; }
                let (nr, nc) = (nr as usize, nc as usize);
                // 0 cost if arrow already points this direction, 1 otherwise
                let cost = if grid[r][c] == (d + 1) as i32 { 0 } else { 1 };
                let nd = dist[r][c] + cost;
                if nd < dist[nr][nc] {
                    dist[nr][nc] = nd;
                    if cost == 0 { dq.push_front((nr, nc)); }
                    else { dq.push_back((nr, nc)); }
                }
            }
        }
        dist[rows-1][cols-1]
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `grid=[[1,1,1,1],[2,2,2,2],[1,1,1,1],[2,2,2,2]]`
Following existing arrows: row 0 all right, row 1 all left, etc. A valid top-to-bottom path requires at most 3 direction changes (crossing each row). Answer: **3**.

### 2. Slightly Complex
**Input:** `grid=[[1,2],[4,3]]`
From (0,0): right to (0,1) costs 0; down to (1,1) costs 1 (arrow=3 not 2). Total: **1**.

### 3. Edge Case: Time Factor
**Input:** $100 \times 100$ grid where every cell points away from the destination.
0-1 BFS still visits each of the $10^4$ cells at most once ($O(nm)$), whereas Dijkstra would process $O(nm \log(nm))$ queue operations.

### 4. Edge Case: Space Factor
**Input:** $100 \times 100$ grid, all cells pointing right.
`dist` array uses $O(nm) = O(10^4)$ integers. The deque holds at most $O(nm)$ elements.

### 5. Almost-Impossible but Plausible
**Input:** $1 \times 1$ grid, `grid=[[1]]`.
Already at the destination; 0 changes needed. Answer: **0**. The deque pops the start cell, no neighbours exist, and $dist[0][0] = 0$ is returned.